In [ ]:
"""
Figure: Methods Pipeline (swim-lane diagram)
============================================

Renders the 21-phase ComputationalReview pipeline as a swim-lane diagram,
faithfully reproducing the actor/critic/validator separation defined in
skills/comprev-orchestrator-v29.md (Phase Index).

Lanes (top to bottom):
  - Coordinator   -- routing across phases; gates between phases (dotted rail)
  - DATAML agent  -- mechanical actors (upper sub-row) + validators (lower sub-row)
  - LITREVIEW agent -- scientific actors and blinded critics

Element types (per Phase Index):
  - ACTOR boxes    -- solid coloured by lane (DATAML blue / LITREVIEW orange)
  - CRITIC boxes   -- LITREVIEW phases with an information barrier from the actor;
                      dashed border + diagonal hatch. Phases 6, 8, 12, 16.
  - VALIDATOR boxes -- DATAML phases that re-check an actor's output, with the same
                       information barrier from the actor; dashed border + diagonal
                       hatch. Phases 1V, 2V, 3V, 5V, 7V, 9V, 14V, 15V, 17V, 19V, 20V,
                       plus Phase 21 (validator-only, no separate actor).

Critics and validators share the same visual convention -- they are the two
flavours of info-barriered reviewer (scientific vs mechanical).  Their lane
colour marks which.  Critics sit inline at the phase x in the LITREVIEW lane;
validators sit inline at the phase x in the DATAML lane's lower sub-row,
directly beneath their actor.

Special cases:
  - Phase 1 has TWO actors (LITREVIEW scoping + DATAML materialise), stacked
    at x=1.  Phase 1V is the validator for the DATAML actor.
  - Phase 20a (Methods Ledger Refresh) runs between Phases 19V and 20.
  - Phase 21 has no separate actor (the deployed site is the input).
"""


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# ---------------------------------------------------------------------------
# Authoritative phase table -- (x_slot, label, role, kind, displayed_phase_id)
# ---------------------------------------------------------------------------
ACTORS = [
    ( 1, "Scope",                  "expert", "actor",  "1"),
    ( 1, "Materialise",            "dataml", "actor",  "1"),
    ( 2, "Evidence\nGathering",    "expert", "actor",  "2"),
    ( 3, "Citation\nInfra",        "dataml", "actor",  "3"),
    ( 4, "Scaffold",               "expert", "actor",  "4"),
    ( 5, "Evidence\nCuration",     "dataml", "actor",  "5"),
    ( 6, "Figure\nAudit",          "expert", "critic", "6"),
    ( 7, "Section\nDrafting",      "expert", "actor",  "7"),
    ( 8, "Section\nCritics",       "expert", "critic", "8"),
    ( 9, "Bibliography",           "dataml", "actor",  "9"),
    (10, "Integration",            "expert", "actor",  "10"),
    (11, "Intro /\nConclusion",    "expert", "actor",  "11"),
    (12, "Bookend\nCritic",        "expert", "critic", "12"),
    (13, "Methods",                "dataml", "actor",  "13"),
    (14, "Document\nAssembly",     "dataml", "actor",  "14"),
    (15, "Citation\nTriples",      "dataml", "actor",  "15"),
    (16, "Citation\nVerification", "expert", "critic", "16"),
    (17, "Fix\nPreparation",       "dataml", "actor",  "17"),
    (18, "Fix\nExecution",         "expert", "actor",  "18"),
    (19, "Fix\nApplication",       "dataml", "actor",  "19"),
    (20, "Methods\nLedger\nRefresh","dataml","actor",  "20a"),
    (21, "Repository\nPush",       "dataml", "actor",  "20"),
]

# Validators live in the DATAML lane lower sub-row at the same x as their actor
# (or at their own x for Phase 21 which is validator-only).
# (x_slot, vlabel)
VALIDATORS = [
    ( 1, "1V" ),    # under Phase 1 DATAML materialise actor
    ( 2, "2V" ),    # under Phase 2 LITREVIEW evidence actor
    ( 3, "3V" ),
    ( 5, "5V" ),
    ( 7, "7V" ),    # under Phase 7 LITREVIEW drafting actor
    ( 9, "9V" ),
    (14, "14V"),
    (15, "15V"),
    (17, "17V"),
    (19, "19V"),
    (21, "20V"),    # under Phase 20 Repository Push (x_slot=21)
    (22.4, "Deploy\nPolish (21)"),   # standalone validator-only phase, shifted right for breathing room
]

# ---------------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------------
EDGE = {"coord": "#2e7d32", "dataml": "#1565c0", "expert": "#e65100"}
FILL = {"coord": "#c8e6c9", "dataml": "#bbdefb", "expert": "#ffe0b2"}
VALIDATOR_FILL = "#e3f2fd"
CRITIC_EDGE = "#6a1b9a"
HATCH = "////"
DASH = (0, (4, 2))

# Stacked DATAML lane: actor upper sub-row, validator lower sub-row
LANE_Y = {"coord": 4.6, "dataml_actor": 2.85, "dataml_val": 1.65, "expert": 0.2}
LANE_LABEL = {
    "coord":  "COORDINATOR\n(routing + gates)",
    "dataml": "DATAML AGENT\n(actors  /  validators)",
    "expert": "LITREVIEW AGENT\n(actors  /  critics)",
}
# DATAML lane spans from below validator to above actor
DATAML_LANE_TOP = LANE_Y["dataml_actor"] + 0.75
DATAML_LANE_BOTTOM = LANE_Y["dataml_val"] - 0.75
DATAML_LANE_CENTER = (DATAML_LANE_TOP + DATAML_LANE_BOTTOM) / 2

# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(22, 8.4))
ax.set_xlim(-2.6, 24.0)
ax.set_ylim(-1.4, 6.0)
ax.axis("off")

box_w, box_h = 0.93, 0.92

# Lane backgrounds
# Coordinator
ax.add_patch(FancyBboxPatch(
    (-1.9, LANE_Y["coord"] - 0.65), 25.6, 1.30,
    boxstyle="round,pad=0.02",
    facecolor=FILL["coord"], edgecolor="none", alpha=0.20, zorder=0,
))
ax.text(-1.8, LANE_Y["coord"], LANE_LABEL["coord"],
        ha="left", va="center", fontsize=8.6, weight="bold",
        color=EDGE["coord"], zorder=1)

# DATAML (tall lane, spans two sub-rows)
ax.add_patch(FancyBboxPatch(
    (-1.9, DATAML_LANE_BOTTOM), 25.6, DATAML_LANE_TOP - DATAML_LANE_BOTTOM,
    boxstyle="round,pad=0.02",
    facecolor=FILL["dataml"], edgecolor="none", alpha=0.20, zorder=0,
))
ax.text(-1.8, DATAML_LANE_CENTER, LANE_LABEL["dataml"],
        ha="left", va="center", fontsize=8.6, weight="bold",
        color=EDGE["dataml"], zorder=1)

# LITREVIEW
ax.add_patch(FancyBboxPatch(
    (-1.9, LANE_Y["expert"] - 0.75), 25.6, 1.50,
    boxstyle="round,pad=0.02",
    facecolor=FILL["expert"], edgecolor="none", alpha=0.20, zorder=0,
))
ax.text(-1.8, LANE_Y["expert"], LANE_LABEL["expert"],
        ha="left", va="center", fontsize=8.6, weight="bold",
        color=EDGE["expert"], zorder=1)

# Coordinator gate rail (dotted)
GATE_Y = LANE_Y["coord"] - 0.75
ax.plot([0.5, 22.5], [GATE_Y, GATE_Y], color="#8a8a8a",
        linewidth=0.7, linestyle=(0, (1, 2)), zorder=1)
ax.text(22.7, GATE_Y, "  gate", ha="left", va="center", fontsize=7,
        color="#666", style="italic")

# ---------------------------------------------------------------------------
# Inter-phase arrows (drawn first so boxes overlay)
# Sequence uses the actor lane for the primary connection; phase 1 is
# represented by its LITREVIEW actor for routing.
# ---------------------------------------------------------------------------
primary = [(1,"expert"),(2,"expert"),(3,"dataml_actor"),(4,"expert"),
           (5,"dataml_actor"),(6,"expert"),(7,"expert"),(8,"expert"),
           (9,"dataml_actor"),(10,"expert"),(11,"expert"),(12,"expert"),
           (13,"dataml_actor"),(14,"dataml_actor"),(15,"dataml_actor"),
           (16,"expert"),(17,"dataml_actor"),(18,"expert"),
           (19,"dataml_actor"),(20,"dataml_actor"),(21,"dataml_actor"),
           (22.4,"dataml_val")]   # Phase 21 (deploy polish) sits in validator row

for i in range(len(primary) - 1):
    x1, lane1 = primary[i]
    x2, lane2 = primary[i + 1]
    y1, y2 = LANE_Y[lane1], LANE_Y[lane2]
    rad = 0.0 if lane1 == lane2 else (-0.18 if y2 < y1 else 0.18)
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=0.9, color="#888",
                                connectionstyle=f"arc3,rad={rad}",
                                shrinkA=20, shrinkB=20),
                zorder=2)

# Phase 1 vertical link (LITREVIEW Scope <-> DATAML Materialise, same x)
ax.annotate("", xy=(1, LANE_Y["expert"] + box_h/2),
            xytext=(1, LANE_Y["dataml_actor"] - box_h/2),
            arrowprops=dict(arrowstyle="-", lw=0.7, color="#aaa",
                            linestyle=(0, (2, 2))),
            zorder=2)

# ---------------------------------------------------------------------------
# Helper: draw a phase box at (x, y) with role/kind styling
# ---------------------------------------------------------------------------
def draw_box(x, y, label, role, kind, displayed_num=None):
    if kind == "critic":
        ec, lw, hatch, ls, weight = CRITIC_EDGE, 1.8, HATCH, DASH, "bold"
        fc = FILL[role]
    elif kind == "validator":
        ec, lw, hatch, ls, weight = EDGE["dataml"], 1.5, HATCH, DASH, "regular"
        fc = VALIDATOR_FILL
    else:
        ec, lw, hatch, ls, weight = EDGE[role], 1.5, None, "-", "regular"
        fc = FILL[role]

    ax.add_patch(FancyBboxPatch(
        (x - box_w/2, y - box_h/2), box_w, box_h,
        boxstyle="round,pad=0.03",
        facecolor=fc, edgecolor=ec, linewidth=lw, hatch=hatch, linestyle=ls,
        zorder=3,
    ))
    ax.text(x, y, label, ha="center", va="center", fontsize=8.0,
            zorder=4, weight=weight)
    if displayed_num is not None:
        ax.text(x, y + box_h/2 + 0.08, displayed_num,
                ha="center", va="bottom", fontsize=8.5, weight="bold",
                color=EDGE.get("dataml" if kind == "validator" else role, EDGE[role]),
                zorder=4)

# ---------------------------------------------------------------------------
# Actor / critic boxes
# ---------------------------------------------------------------------------
for x, label, role, kind, num in ACTORS:
    y = LANE_Y["dataml_actor"] if role == "dataml" else LANE_Y["expert"]
    draw_box(x, y, label, role, kind, displayed_num=num)
    # Coordinator pin (only one per slot to avoid double-draw on Phase 1)
    if not (x == 1 and role == "dataml"):
        ax.plot([x, x], [LANE_Y["coord"] + box_h/2 - 0.5, GATE_Y],
                color="#bbb", linewidth=0.5, linestyle=":", zorder=1)

# ---------------------------------------------------------------------------
# Validator boxes (full-size, in DATAML lower sub-row)
# ---------------------------------------------------------------------------
for x, vlabel in VALIDATORS:
    # Display number = vlabel itself (e.g., "1V", "3V", "Deploy Polish (21)")
    if x == 22.4:
        # Phase 21 — validator-only, no actor above it; show full label and number "21"
        draw_box(x, LANE_Y["dataml_val"], vlabel, "dataml", "validator",
                 displayed_num="21")
        ax.plot([x, x], [LANE_Y["coord"] + box_h/2 - 0.5, GATE_Y],
                color="#bbb", linewidth=0.5, linestyle=":", zorder=1)
    else:
        draw_box(x, LANE_Y["dataml_val"], vlabel, "dataml", "validator",
                 displayed_num=None)
        # Connector line from validator up to its actor (LITREVIEW for 2V/7V,
        # DATAML for the others)
        if vlabel in ("2V", "7V"):
            actor_y = LANE_Y["expert"]
        else:
            actor_y = LANE_Y["dataml_actor"]
        ax.plot([x, x],
                [LANE_Y["dataml_val"] + box_h/2, actor_y - box_h/2],
                color="#9e9e9e", linewidth=0.6, linestyle=":", zorder=1)

# ---------------------------------------------------------------------------
# Footer caveat
# ---------------------------------------------------------------------------
ax.text(0.5, -1.20,
        "Phase 1 has two simultaneous actors (LITREVIEW scoping + DATAML "
        "materialisation, dashed link).  Phase 20a (Methods Ledger Refresh) "
        "runs between Phases 19V and 20.  Phase 21 (Deploy Polish) has no "
        "separate actor -- its actor and validator are the same frame.",
        ha="left", va="top", fontsize=7.2, color="#555", style="italic")

# ---------------------------------------------------------------------------
# Legend
# ---------------------------------------------------------------------------
legend_items = [
    mpatches.Patch(facecolor=FILL["coord"], edgecolor=EDGE["coord"],
                   label="Coordinator -- routing + gates"),
    mpatches.Patch(facecolor=FILL["dataml"], edgecolor=EDGE["dataml"],
                   label="DATAML actor -- mechanical phase"),
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=EDGE["expert"],
                   label="LITREVIEW actor -- scientific phase"),
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=CRITIC_EDGE,
                   linestyle="--", hatch=HATCH,
                   label="LITREVIEW critic -- info-barriered review (scientific)"),
    mpatches.Patch(facecolor=VALIDATOR_FILL, edgecolor=EDGE["dataml"],
                   linestyle="--", hatch=HATCH,
                   label="DATAML validator -- info-barriered review (mechanical)"),
]
ax.legend(handles=legend_items, loc="lower center",
          bbox_to_anchor=(0.5, -0.12), ncol=3,
          frameon=False, fontsize=8.5, handlelength=2.0, handleheight=1.1)

ax.set_title(
    "Expert Review Pipeline v29 -- actor / reviewer separation across 21 phases (+ 20a)",
    fontsize=12.5, weight="bold", pad=10,
)

plt.subplots_adjust(left=0.02, right=0.99, top=0.93, bottom=0.13)

out_path = os.path.join(
    os.path.dirname(os.path.abspath("__nb__")), "..", "fig_methods_pipeline.png"
)
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
print("saved:", out_path)
